# Clase 1 — MLOps y buenas prácticas de programación

**Objetivo:** entender qué es MLOps y por qué el código del TP de AMq1 (notebooks) tuvo que
convertirse en un **paquete** con funciones, docstrings y tests antes de llevarlo a producción.

Temas de la clase 1 de la materia: ciclo de vida de un proyecto de ML · MLOps y sus niveles ·
buenas prácticas (código limpio, modularidad, docstrings, PEP 8, linters, tests).

In [1]:
from helpers import OUT, ROOT, check, cuestionario, quiz, safe_check, tarjetas

import inspect
import subprocess
import sys

import pandas as pd
from ipywidgets import interact, widgets

## 📖 Teoría: el ciclo de vida de un modelo y dónde entra MLOps

Un proyecto de ML no termina cuando el modelo da buena métrica. El ciclo completo es circular:

```mermaid
flowchart LR
    A["1. Problema<br>y datos"] --> B["2. Preparación<br>de datos"]
    B --> C["3. Entrenamiento<br>y evaluación"]
    C --> D["4. Despliegue"]
    D --> E["5. Monitoreo"]
    E -- "los datos cambian<br>(drift)" --> B
```

En AMq1 hicimos los pasos 1 a 3 **a mano**. MLOps se ocupa de que los pasos 2 a 5 sean
**automáticos, reproducibles y observables**. Une tres disciplinas: ML (modelos), ingeniería de
software (código mantenible, tests) y operaciones (infraestructura, despliegue, monitoreo).

**Por qué hace falta.** El paper *Hidden Technical Debt in Machine Learning Systems* (Sculley et al.,
Google, NeurIPS 2015) mostró que en un sistema real **el código del modelo es una caja chica** rodeada
de cajas mucho más grandes: recolección y verificación de datos, configuración, extracción de features,
infraestructura de servicio, monitoreo. Esa parte es la que acumula deuda técnica.

**Los pilares que vamos a ver en la materia:**

| Pilar | Pregunta que responde | Herramienta en el TP |
|---|---|---|
| Código de calidad | ¿otro puede entender y modificar esto? | paquete `src/burnout`, `ruff`, docstrings |
| Tests | ¿cómo sé que no rompí nada? | `pytest` |
| Reproducibilidad | ¿puedo volver a obtener el mismo modelo? | semilla, hilos, versiones, Docker |
| Seguimiento de experimentos | ¿qué probé y qué funcionó? | MLflow |
| Orquestación | ¿quién corre el pipeline y cuándo? | Airflow |
| Despliegue | ¿cómo llegan las predicciones al usuario? | DAG batch, FastAPI (2ª entrega) |

**Buenas prácticas de código, en una frase cada una:**

- **Modularidad:** funciones chicas con una sola responsabilidad (`validate`, `make_split`, `make_pipeline`).
- **DRY:** cada constante y cada regla en un solo lugar (`config.py`).
- **Nombres que explican:** `make_split(df)` en lugar de `f(d)`.
- **Docstrings y *type hints*:** el contrato de la función, escrito.
- **PEP 8 + linter:** estilo uniforme sin discutirlo en cada revisión.
- **Tests:** la red de seguridad para cambiar código sin miedo.

## 1. ¿Qué es MLOps?
Un modelo en un notebook **no es un producto**. MLOps es el conjunto de prácticas para que un modelo
llegue a producción y se mantenga ahí: datos versionados, entrenamientos reproducibles, despliegue,
monitoreo y reentrenamiento.

Niveles de madurez (Google):

| Nivel | Cómo se trabaja |
|---|---|
| **0** | Todo manual: notebooks, el modelo se pasa "a mano" a quien lo despliega |
| **1** | Pipeline de entrenamiento automatizado (orquestador), entrenamiento continuo |
| **2** | CI/CD del propio pipeline: tests y despliegue automático de cada cambio |

In [2]:
quiz(
    "El TP de AMq1 (7 notebooks que se corren a mano en orden), ¿en qué nivel está?",
    ["Nivel 0", "Nivel 1", "Nivel 2"],
    answer=0,
    explanation="Todo es manual. Con el DAG de Airflow y MLflow de este TP pasamos a nivel 1.",
)

**El TP de AMq1 (7 notebooks que se corren a mano en orden), ¿en qué nivel está?**

ToggleButtons(options=('Nivel 0', 'Nivel 1', 'Nivel 2'), style=ToggleButtonsStyle(button_width='auto'), value=…

Output()

## 2. Del notebook al paquete
En AMq1 cada notebook copiaba al principio las mismas celdas: constantes, carga, preprocesamiento.
Eso viola **DRY** (*Don't Repeat Yourself*): un cambio hay que hacerlo en siete lugares.

Ahora todo vive en `src/burnout/`:

In [3]:
for path in sorted((ROOT / "src" / "burnout").glob("*.py")):
    primera_linea = path.read_text(encoding="utf-8").split('"""')[1].strip().splitlines()[0]
    print(f"{path.name:<12} {primera_linea}")

__init__.py  Riesgo de burnout de estudiantes: código del modelo de AMq1 listo para producción.
batch.py     Predicción por lotes. El DAG ``batch_predict_burnout`` usa estas funciones.
config.py    Constantes del proyecto: esquema del dataset, features, semilla y nombres de MLflow/S3.
data.py      Carga, validación y split del dataset.
etl.py       Pasos del ETL. El DAG ``etl_burnout`` llama a estas funciones desde sus tareas.
model.py     Preprocesamiento, pipeline de XGBoost y métricas.
storage.py   Lectura y escritura en el Data Lake (MinIO, compatible con S3) usando boto3.
train.py     Entrenamiento del modelo final y registro opcional en MLflow.


### 🎛️ Explorá el código del paquete
Elegí una función para ver su código y su docstring. Fijate en: nombres descriptivos, una sola
responsabilidad por función, docstring con parámetros y retorno, *type hints*.

In [4]:
import burnout.data
import burnout.model
import burnout.storage
import burnout.train

funciones = {
    f"{mod.__name__}.{name}": obj
    for mod in [burnout.data, burnout.model, burnout.storage, burnout.train]
    for name, obj in inspect.getmembers(mod, inspect.isfunction)
    if obj.__module__ == mod.__name__ and not name.startswith("_")
}


@interact(funcion=widgets.Dropdown(options=sorted(funciones), value="burnout.data.make_split"))
def ver_codigo(funcion):
    print(inspect.getsource(funciones[funcion]))

interactive(children=(Dropdown(description='funcion', index=3, options=('burnout.data.decode_target', 'burnout…

### ✏️ Ejercicio 1 — refactorizá una celda "de notebook"
Este código está escrito como en un notebook apurado:

```python
d = pd.read_csv(ruta)
d["t"] = d["Weekly_GenAI_Hours"] + d["Traditional_Study_Hours"]
d["p"] = d["Weekly_GenAI_Hours"] / d["t"]
```

Escribí la función `proporcion_ia(df)` que devuelva una **Serie** con la proporción de horas de
estudio que se hacen con IA (horas IA / horas totales), con un docstring y *type hints*.

In [5]:
def proporcion_ia(df: pd.DataFrame) -> pd.Series:
    """Escribí acá el docstring."""
    ...  # ← tu código acá

In [6]:
from burnout.config import DATA_RAW

df = pd.read_csv(DATA_RAW)
esperado = df["Weekly_GenAI_Hours"] / (df["Weekly_GenAI_Hours"] + df["Traditional_Study_Hours"])
safe_check(
    "Ejercicio 1 (resultado)",
    lambda: ((proporcion_ia(df) - esperado).abs() < 1e-12).all(),
    "`ia = df['Weekly_GenAI_Hours']` y devolvé `ia / (ia + df['Traditional_Study_Hours'])`.",
)
check(
    "Ejercicio 1 (docstring)",
    bool(proporcion_ia.__doc__) and "Escribí" not in proporcion_ia.__doc__,
    "reemplazá el docstring por una explicación de qué devuelve la función.",
)

⏳ **Ejercicio 1 (resultado)**: todavía no. *Pista:* `ia = df['Weekly_GenAI_Hours']` y devolvé `ia / (ia + df['Traditional_Study_Hours'])`. (error: `TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'`)

⏳ **Ejercicio 1 (docstring)**: todavía no. *Pista:* reemplazá el docstring por una explicación de qué devuelve la función.

False

<details><summary>Ver solución</summary>

```python
def proporcion_ia(df: pd.DataFrame) -> pd.Series:
    """Proporción de las horas semanales de estudio que el estudiante hace con IA (0 a 1)."""
    ia = df["Weekly_GenAI_Hours"]
    return ia / (ia + df["Traditional_Study_Hours"])
```
</details>

## 3. Tests automáticos
Un test es una función que verifica una propiedad del código. Si mañana alguien cambia el split o
el preprocesamiento por error, el test avisa **antes** de que llegue a producción.

Los tests del TP están en `tests/`. Corrámoslos (tarda unos segundos):

In [7]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests")], capture_output=True,
                   text=True, cwd=ROOT)
print(r.stdout[-1500:])

.  This option produces incorrect results by returning ``None`` for related items. (deprecated since: 2.1)
    sqlalchemy.orm.configure_mappers()

tests/test_model.py::test_log_and_register_en_mlflow_local
  C:\Users\USUARIO\CEIA\MLOps-TP-AVS\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
    warnings.warn(

tests/test_model.p

Un test que vale la pena mirar es `test_split_sin_fuga_y_reproducible`: verifica que **ninguna
variable de resultado** quede entre las features (la fuga de información del repaso R1).

### ✏️ Ejercicio 2 — escribí tu propio test
Completá `test_proporcion_entre_0_y_1` para que verifique que `proporcion_ia(df)` siempre está entre
0 y 1. Un test usa `assert`: si la condición es falsa, falla.

In [8]:
def test_proporcion_entre_0_y_1():
    p = proporcion_ia(df)
    ...  # ← tu assert acá

In [9]:
def _corre_y_detecta_error():
    test_proporcion_entre_0_y_1()  # tiene que pasar...
    roto = lambda d: d["Weekly_GenAI_Hours"]  # noqa: E731  ...y fallar con una función rota
    global proporcion_ia
    original, proporcion_ia = proporcion_ia, roto
    try:
        test_proporcion_entre_0_y_1()
        return False  # el test no detectó el error
    except AssertionError:
        return True
    finally:
        proporcion_ia = original


safe_check(
    "Ejercicio 2",
    _corre_y_detecta_error,
    "`assert p.between(0, 1).all()`. (También hay que haber resuelto el ejercicio 1.)",
)

⏳ **Ejercicio 2**: todavía no. *Pista:* `assert p.between(0, 1).all()`. (También hay que haber resuelto el ejercicio 1.)

False

<details><summary>Ver solución</summary>

```python
def test_proporcion_entre_0_y_1():
    p = proporcion_ia(df)
    assert p.between(0, 1).all()
```
</details>

## 4. Estilo: PEP 8 y linters
Un **linter** revisa el código sin ejecutarlo. En el TP usamos **ruff** (reemplaza a pylint, flake8
e isort, y su `ruff format` reemplaza a autopep8/black). Mirá qué encuentra en este código:

In [10]:
codigo_feo = '''import os, sys
import pandas as pd
def Calcular(x,y):
    resultado=x+y
    return x+y
'''
archivo = OUT / "feo.py"
archivo.write_text(codigo_feo, encoding="utf-8")
r = subprocess.run([sys.executable, "-m", "ruff", "check", "--select", "E,F,I,N", "--no-cache", str(archivo)],
                   capture_output=True, text=True)
print(r.stdout)

E401 [*] Multiple imports on one line
 --> _salidas\feo.py:1:1
  |
1 | import os, sys
  | ^^^^^^^^^^^^^^
2 | import pandas as pd
3 | def Calcular(x,y):
  |
help: Split imports
  |
  - import os, sys
1 + import os
2 + import sys
3 | import pandas as pd
  |

I001 [*] Import block is un-sorted or un-formatted
 --> _salidas\feo.py:1:1
  |
1 | / import os, sys
2 | | import pandas as pd
  | |___________________^
3 |   def Calcular(x,y):
4 |       resultado=x+y
  |
help: Organize imports
  |
  - import os, sys
1 + import os
2 + import sys
3 +
4 | import pandas as pd
5 +
6 +
7 | def Calcular(x,y):
  |

F401 [*] `os` imported but unused
 --> _salidas\feo.py:1:8
  |
1 | import os, sys
  |        ^^
2 | import pandas as pd
3 | def Calcular(x,y):
  |
help: Remove unused import
  |
  - import os, sys
1 | import pandas as pd
  |

F401 [*] `sys` imported but unused
 --> _salidas\feo.py:1:12
  |
1 | import os, sys
  |            ^^^
2 | import pandas as pd
3 | def Calcular(x,y):
  |
help: Remove unuse

In [11]:
quiz(
    "¿Qué significa F841 'Local variable `resultado` is assigned to but never used'?",
    ["Un error de sintaxis", "Una variable que se calcula y nunca se usa", "Una importación faltante"],
    answer=1,
    explanation="Suele delatar un bug: calculamos `resultado` pero devolvimos otra cosa.",
)

**¿Qué significa F841 'Local variable `resultado` is assigned to but never used'?**

ToggleButtons(options=('Un error de sintaxis', 'Una variable que se calcula y nunca se usa', 'Una importación …

Output()

Para revisar y formatear todo el TP (desde la raíz del repo):

```bash
ruff check .
ruff format .
```

## 5. Reproducibilidad: un caso real de este TP
En AMq1 el F1 macro de test del XGBoost fue **0,5405**. Al pasar el código al paquete obtuvimos
**0,5372** con el mismo código, los mismos datos y la misma semilla. ¿Qué pasó?

XGBoost con `tree_method="hist"` suma histogramas en paralelo, y **el orden de las sumas de punto
flotante cambia con la cantidad de hilos**. En AMq1 usamos `n_jobs=-1` (todos los núcleos de esa
máquina). Probalo:

In [12]:
from burnout.config import BEST_PARAMS_AMQ1
from burnout.data import load_raw, make_split
from burnout.model import make_pipeline, test_metrics

X_train, X_test, y_train, y_test = make_split(load_raw())
probados = {}


@interact(n_jobs=widgets.SelectionSlider(options=[1, 2, 4, 8, 12, 16], value=4), manual=True,
          manual_name="Entrenar")
def probar_hilos(n_jobs):
    m = make_pipeline(**{**BEST_PARAMS_AMQ1, "n_jobs": n_jobs}).fit(X_train, y_train)
    probados[n_jobs] = test_metrics(y_test, m.predict(X_test))["f1_macro"]
    print(pd.Series(probados, name="F1 macro test").sort_index().round(4).to_string())

interactive(children=(SelectionSlider(description='n_jobs', index=2, options=(1, 2, 4, 8, 12, 16), value=4), O…

Las diferencias son chicas (todas caen dentro del intervalo de confianza de AMq1), pero en
producción importan: **un contenedor Docker con otra cantidad de CPUs daría otro modelo**. Por eso
fijamos `XGB_N_JOBS = 4` en `burnout/config.py` y el test `test_entrenamiento_determinista` lo vigila.

Lista de control de reproducibilidad del TP:

- [x] semilla fija (`SEED = 42`) en split, CV, Optuna y XGBoost
- [x] cantidad de hilos fija
- [x] versiones exactas de librerías (`requirements.txt`, y las mismas en las imágenes de Docker)
- [x] datos versionados (CSV en el repo, con su SHA-256 en `data/README.md`)

## 6. Trabajo en equipo con Git
El TP es grupal y la materia evalúa la dinámica de equipo. Flujo sugerido:

```bash
git switch -c feature/dag-etl        # una rama por tarea
# ... cambios ...
ruff check . && pytest               # antes de commitear
git add -p && git commit -m "Agrego DAG de ETL"
git push -u origin feature/dag-etl   # y abrir un Pull Request para que otro lo revise
```

In [13]:
quiz(
    "Victoria y Alexis editan el mismo DAG a la vez en `main`. ¿Qué conviene?",
    ["Que uno pise el trabajo del otro", "Cada uno en su rama y unir con Pull Requests", "Mandarse el archivo por WhatsApp"],
    answer=1,
    explanation="Las ramas aíslan el trabajo y el PR deja revisar los cambios antes de unirlos.",
)

**Victoria y Alexis editan el mismo DAG a la vez en `main`. ¿Qué conviene?**

ToggleButtons(options=('Que uno pise el trabajo del otro', 'Cada uno en su rama y unir con Pull Requests', 'Ma…

Output()

## 🏭 Casos de uso y ejemplos

### En la industria

- **Uber · Michelangelo (2017).** Antes, cada equipo de Uber armaba su propio camino para entrenar y
  desplegar modelos, y pocos llegaban a producción. Michelangelo unificó el flujo (datos → entrenamiento
  → evaluación → despliegue → monitoreo) en una plataforma. Es el mismo recorrido que hace este TP, en
  chico.
- **Zillow Offers (2021).** Zillow compraba casas según el precio que estimaban sus modelos. Cuando
  el mercado cambió, los modelos sobrestimaron precios, la empresa cerró la unidad y registró pérdidas
  de cientos de millones de dólares. La lección de MLOps: **monitorear** el modelo en producción y
  detectar cuándo el mundo dejó de parecerse a los datos de entrenamiento.
- **Reproducibilidad en este mismo TP.** Al pasar el código de AMq1 a un paquete descubrimos que el
  F1 cambiaba según la cantidad de hilos de XGBoost (§5). Sin tests, ese tipo de diferencia aparece
  recién en producción.

### Casos prácticos

**Caso 1.** Victoria entrena el modelo en su laptop y le da F1 0,5372; a vos te da 0,5365 con el
mismo código. ¿Qué revisan?

<details><summary>Respuesta</summary>

Las cuatro fuentes de no reproducibilidad: (1) **versiones** de librerías (`pip freeze` contra
`requirements.txt`), (2) **semillas**, (3) **hilos** (`n_jobs`) y (4) **datos** (el SHA-256 del CSV en
`data/README.md`). En el TP ya fijamos las cuatro, así que tendría que dar igual en las dos laptops.
</details>

**Caso 2.** Alexis agrega una feature nueva en un notebook, pero el DAG de entrenamiento sigue usando
la lista vieja. ¿Qué principio se violó y cómo se evita?

<details><summary>Respuesta</summary>

**DRY**: la lista de features estaba duplicada. Se evita usando siempre `FEATURE_COLUMNS` de
`burnout/config.py` desde el notebook, el DAG y los tests.
</details>

**Caso 3.** Alguien cambia `validate()` y, sin querer, deja de detectar nulos. ¿Cómo se habría enterado
el equipo antes de que llegue a `main`?

<details><summary>Respuesta</summary>

Con `tests/test_data.py`: hay un test que mete un nulo y espera `ValueError`. Si se corre `pytest`
antes de cada commit (o en CI sobre cada Pull Request), el cambio no pasa la revisión.
</details>

### Ejemplos de uso de las herramientas

```bash
pytest                         # todos los tests
pytest tests/test_data.py -v   # un archivo, mostrando cada test
pytest -k determinista         # solo los tests cuyo nombre contiene "determinista"
pytest -x                      # frena en el primer fallo
ruff check .                   # revisa estilo y errores comunes
ruff check . --fix             # corrige solo lo que se puede corregir automáticamente
```

## Resumen
- MLOps = llevar y mantener modelos en producción; el TP nos lleva del nivel 0 al 1.
- El código vive en un paquete (`src/burnout`) con funciones chicas, docstrings y *type hints*.
- Tests (`pytest`) y linter (`ruff`) cuidan que nada se rompa sin que nos demos cuenta.
- Reproducible = semilla + hilos + versiones + datos fijos.

Siguiente: **C2 — Docker**.

## 🔁 Repaso con tarjetas
Leé el concepto, intentá explicarlo con tus palabras y recién después apretá **Mostrar respuesta**.
Marcá **La sabía** o **A repasar**: al final te queda la lista de lo que conviene volver a mirar.

In [14]:
tarjetas("C1")

## 📝 Cuestionario final
Respondé todas y apretá **Corregir**. Cada respuesta trae su explicación. Meta: 7 de 8 o más.

In [15]:
cuestionario("C1")